# Session 3–4 Lab: Simulating HDFS and MapReduce with QuickKart Order Data

**Course:** Big Data Analytics — MBA BA, Trim IV
**Business Case:** QuickKart, a grocery delivery app operating across three regions (North, South, West)

---

### Why this notebook exists

You won't have a real multi-machine Hadoop cluster in front of you for most of the term, but the *ideas* behind HDFS and MapReduce don't require one to understand. This notebook rebuilds both concepts using nothing but Python, pandas, and your laptop's own CPU cores — so you can see blocks, replication, a NameNode, and a Map → Shuffle → Reduce pipeline actually run, instead of just reading about them.

### The business problem we'll solve

QuickKart's order data is generated in three regional data centers (North, South, West) as the company grows too large for one server to hold it all. Corporate HQ wants one question answered:

> **"Which product categories are driving the most revenue across the whole business, and can we get that answer reliably even if one regional server goes down?"**

This single business question is exactly what HDFS (reliable distributed storage) and MapReduce (distributed computation) were built to solve — we'll solve it twice: once the "distributed systems" way, and once by comparing it to what a single machine doing all the work would look like.


## Part 0: Generate QuickKart Order Data

We'll simulate a realistic-looking order dataset — think of this as "yesterday's orders" across all of QuickKart's regions. In a real company this would already be sitting in a database or data lake; here we generate it so the notebook is self-contained.


In [ ]:
import pandas as pd
import numpy as np
import os, shutil, time
from concurrent.futures import ThreadPoolExecutor

np.random.seed(42)

N_ORDERS = 12000

categories = ["Fruits & Vegetables", "Dairy", "Snacks", "Beverages",
              "Household", "Personal Care", "Bakery", "Frozen Foods"]

regions = ["North", "South", "West"]

df = pd.DataFrame({
    "OrderID": range(1, N_ORDERS + 1),
    "Region": np.random.choice(regions, N_ORDERS, p=[0.4, 0.35, 0.25]),
    "Category": np.random.choice(categories, N_ORDERS),
    "OrderValue": np.round(np.random.gamma(shape=3.5, scale=120, size=N_ORDERS), 2),
    "Timestamp": pd.date_range("2026-01-01", periods=N_ORDERS, freq="4min")
})

df.head()

**What we just built:** 12,000 rows representing individual QuickKart orders, each tagged with a region, a product category, and an order value. In real life, this file alone might be several GB per day once you include every line item, delivery metadata, and customer info — easily large enough that no single laptop's RAM comfortably holds a year of it. That's the scale problem HDFS and MapReduce exist to solve.


## Part 1: Simulating HDFS — Blocks, Replication, and a NameNode

### The warehouse analogy (recap)

- **Blocks** — HDFS doesn't store a file as one giant blob. It slices it into fixed-size chunks (128MB by default in real Hadoop). Here, we'll slice our order data into 3 chunks.
- **DataNodes** — the "shelves" where blocks physically live. We'll simulate 3 DataNodes as 3 folders on disk.
- **Replication** — each block is copied onto more than one DataNode, so losing one machine doesn't lose data. We'll use a replication factor of 2.
- **NameNode** — the "warehouse manager" that doesn't store any actual data, just the map of which block lives on which DataNode(s). We'll represent this as a Python dictionary.

### Business framing

Picture QuickKart's order data physically split across regional servers because no single server can hold the whole company's transaction history. If the North region's server has a disk failure at 2am, corporate reporting shouldn't break — this section demonstrates exactly why replication makes that true.


In [ ]:
# Clean slate
if os.path.exists("cluster"):
    shutil.rmtree("cluster")

for node in ["datanode1", "datanode2", "datanode3"]:
    os.makedirs(f"cluster/{node}", exist_ok=True)

# 1. Split the dataset into 3 "blocks"
block_size = len(df) // 3
block1 = df.iloc[0:block_size]
block2 = df.iloc[block_size:2*block_size]
block3 = df.iloc[2*block_size:]

blocks = {"block1.csv": block1, "block2.csv": block2, "block3.csv": block3}

for name, chunk in blocks.items():
    print(f"{name}: {len(chunk)} rows")


Notice we split by *row ranges*, not by region or category. That's realistic — HDFS has no idea what's *inside* a file when it splits it into blocks; it just cuts by size. Any business meaning (which region, which category) only gets applied later, during Map/Reduce. This is an important distinction for students: **storage layer is dumb and mechanical; meaning is applied by the compute layer on top.**


In [ ]:
# 2. Write each block to TWO DataNode folders (replication factor = 2)
placement = {
    "block1.csv": ["datanode1", "datanode2"],
    "block2.csv": ["datanode2", "datanode3"],
    "block3.csv": ["datanode3", "datanode1"],
}

for block_name, nodes in placement.items():
    for node in nodes:
        blocks[block_name].to_csv(f"cluster/{node}/{block_name}", index=False)

# 3. The "NameNode" — just metadata, no actual data
namenode_metadata = placement
namenode_metadata


**This dictionary *is* a NameNode**, conceptually. A real Hadoop NameNode stores far more (permissions, timestamps, exact byte offsets), but the essential job — "which block lives on which machine(s)" — is exactly what you're looking at above.


In [ ]:
# Verify: list what's physically sitting in each DataNode folder
for node in ["datanode1", "datanode2", "datanode3"]:
    files = os.listdir(f"cluster/{node}")
    print(f"{node}: {files}")


### Fault-tolerance demo — kill a DataNode

We'll now simulate DataNode1 failing (disk crash, network partition, power outage — doesn't matter which). Watch what happens when we try to read `block1.csv`, which lived on both DataNode1 *and* DataNode2.


In [ ]:
def read_block(block_name, metadata):
    """Simulates what a real HDFS client does: ask the NameNode where a
    block lives, then try each replica location until one succeeds."""
    for node in metadata[block_name]:
        path = f"cluster/{node}/{block_name}"
        if os.path.exists(path):
            print(f"Read '{block_name}' successfully from {node}")
            return pd.read_csv(path)
    raise FileNotFoundError(f"All replicas of {block_name} are unavailable!")

# Simulate DataNode1 going down
os.remove("cluster/datanode1/block1.csv")
os.remove("cluster/datanode1/block3.csv")
print("DataNode1 has 'failed' — its files are gone.\n")

# Try reading block1 anyway
recovered = read_block("block1.csv", namenode_metadata)
recovered.head()


**What just happened, in business terms:** DataNode1 (say, QuickKart's North region server) went offline entirely. Corporate's revenue report didn't fail — it silently fell back to the replica on DataNode2. This is precisely why production Hadoop clusters default to a replication factor of 3: it's not about backup for backup's sake, it's about a live system continuing to serve business-critical queries during hardware failure, without anyone needing to intervene at 2am.

**Discussion prompt for class:** What would happen instead if the *NameNode* — not a DataNode — went down? (Answer: the whole cluster becomes unreadable, because no client can figure out where any block lives, even though the data itself is physically fine. This is why production clusters run NameNode High Availability with a standby.)


## Part 2: Demoing MapReduce — Answering the Business Question

Recall the question from corporate HQ:

> **Which product categories are driving the most revenue across the whole business?**

We'll answer it using the Map → Shuffle → Reduce pattern, computing partial results independently on each block (as if each ran on a different machine), then combining them.

### The three phases

| Phase | What it does here | Real Hadoop equivalent |
|---|---|---|
| **Map** | Each block computes its own category totals, independently | Mapper tasks run *where the data already sits* — no data movement |
| **Shuffle** | Partial results from all blocks get grouped by key (Category) | Hadoop physically moves/sorts intermediate key-value pairs across the network |
| **Reduce** | Grouped partial results get summed into final totals | Reducer tasks aggregate each key's values into one final answer |


In [ ]:
# MAP step: this function is what would run independently on each "node"
def map_function(filepath):
    chunk = pd.read_csv(filepath)
    partial = chunk.groupby("Category")["OrderValue"].agg(order_count="count", revenue="sum")
    return partial

# Point at one surviving copy of each block (simulating 3 nodes each doing their own slice)
block_paths = [
    "cluster/datanode2/block1.csv",   # block1's surviving replica
    "cluster/datanode2/block2.csv",
    "cluster/datanode3/block3.csv",   # block3's surviving replica
]

block_paths


### Running Map in parallel (this is the "distributed" part made visible)

We'll use Python's `multiprocessing.Pool` to genuinely run the three Map tasks on separate CPU cores at the same time — not just sequentially in a loop. This is the closest a single laptop can get to simulating what happens across three physical machines in a real cluster.


In [ ]:
# Parallel execution across 3 simulated nodes using threading
start = time.time()
with ThreadPoolExecutor(max_workers=3) as executor:
    map_results = list(executor.map(map_function, block_paths))
parallel_time = time.time() - start

print(f"Parallel Map phase across 3 simulated nodes took {parallel_time:.4f} seconds")
for i, result in enumerate(map_results, start=1):
    print(f"\n--- Partial result from node {i} ---")
    print(result)

In [ ]:
# Compare: what if ONE machine had to process the entire dataset sequentially?
start = time.time()
sequential_result = df.groupby("Category")["OrderValue"].agg(order_count="count", revenue="sum")
sequential_time = time.time() - start

print(f"Single-machine sequential processing took {sequential_time:.4f} seconds")
print(f"\nSpeedup from distributing across 3 simulated nodes: {sequential_time/parallel_time:.2f}x")


**Be honest with students about this number.** On a small 12,000-row dataset on a laptop, the "speedup" may be modest or even negative — the overhead of spinning up separate processes can outweigh the benefit at small scale. That's a genuine and important lesson: **distributed computing has overhead, and it only pays off once data size crosses a threshold.** This is exactly why companies don't put every dataset in Hadoop — a bloated Excel file doesn't need a cluster, but QuickKart's full transaction history across every region for the past 3 years absolutely does.


In [ ]:
# SHUFFLE + REDUCE: combine partial results from all nodes by key (Category)
combined = pd.concat(map_results)
final_result = combined.groupby(combined.index).sum().sort_values("revenue", ascending=False)

final_result


### Sanity check — does our distributed answer match the single-machine answer?

This step matters pedagogically: MapReduce should be *transparent* to the business user. The person reading the report shouldn't be able to tell whether it was computed on one machine or a thousand — the answer must be identical either way.


In [ ]:
comparison = pd.DataFrame({
    "Distributed (Map-Shuffle-Reduce)": final_result["revenue"],
    "Single-machine (direct groupby)": sequential_result.sort_values("revenue", ascending=False)["revenue"],
})
comparison["Match"] = np.isclose(comparison["Distributed (Map-Shuffle-Reduce)"], comparison["Single-machine (direct groupby)"])
comparison


## Part 3: Visualizing the Business Answer


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 5))
final_result["revenue"].sort_values().plot(kind="barh", ax=ax, color="#1F4E79")
ax.set_xlabel("Total Revenue (₹)")
ax.set_ylabel("")
ax.set_title("QuickKart: Revenue by Category (Combined Across All Regions)")
plt.tight_layout()
plt.show()


## Recap: Mapping Concepts Back to Real Hadoop

| What we built | Real Hadoop / Spark equivalent |
|---|---|
| `cluster/datanode1/`, `datanode2/`, `datanode3/` folders | DataNodes storing HDFS blocks |
| `block1.csv`, `block2.csv`, `block3.csv` | HDFS blocks (128MB chunks in production) |
| `placement` dictionary | NameNode metadata |
| Writing each block to 2 folders | HDFS replication factor |
| Deleting a file to simulate failure | A DataNode crashing |
| `read_block()` trying multiple locations | HDFS client failover to a healthy replica |
| `map_function()` | A Mapper task |
| `pool.map()` running 3 functions concurrently | Parallel task execution across a YARN-managed cluster |
| `pd.concat()` + `groupby().sum()` | Shuffle + Reduce phases |

### Discussion questions for class

1. At what dataset size do you think the "distributed" approach would start clearly beating the single-machine approach on your own laptop? How would you test that?
2. If QuickKart added a 4th region tomorrow, what would need to change in our simulation — and what would change in a real Hadoop cluster?
3. Our NameNode was a single Python dictionary that can't fail. What does that hide about the real-world NameNode single-point-of-failure problem?
4. We used replication factor 2. Real Hadoop defaults to 3. What business trade-off is being made by choosing a lower or higher replication factor (storage cost vs. fault tolerance)?

### Business takeaway

QuickKart didn't adopt distributed storage and computation because it's technically fashionable — it did so because **a single server both ran out of room to store the data and ran out of time to process it fast enough**, and because **corporate reporting can't go dark just because one regional server has a bad night.** Everything you built in this notebook, from block placement to the Map/Shuffle/Reduce pipeline, exists to solve those two very ordinary business problems at scale.
